<img src= "https://cdn.oreillystatic.com/images/sitewide-headers/oreilly_logo_mark_red.svg"/>&nbsp;&nbsp;<font size="16"><b>AI, ML and GenAI in the Lakehouse<b></font></span>
<img style="float: left; margin: 0px 15px 15px 0px; width:30%; height: auto;" src="https://i.imgur.com/FWzhbhX.jpeg"   />

   Name:          chapter 04-04 Model Predictions

   Author:    Bennie Haelen
   Date:      10-12-2024

   Purpose:   This notebook demonstrates three ways to load and run predictions
              with the registered model from the model building notebook.

      An outline of the different sections in this notebook:
        1 - Read the feature table from Unity Catalog and prepare test data
              1-1 Read the feature table
              1-2 Define the feature matrix (X) and target variable (y)
              1-3 Split into training and test sets
              1-4 Apply standard scaling
        2 - Load the model from the Unity Catalog registry (recommended)
              2-1 Build the model URI from the registry
              2-2 Load the model as a native scikit-learn model
              2-3 Run predictions and evaluate results
        3 - Load the model as a Spark UDF
              3-1 Create the Spark UDF from the model URI
              3-2 Create a Spark DataFrame from the scaled test values
              3-3 Run predictions with the Spark UDF
              3-4 Verify the predictions
        4 - Load the model as a generic PyFunc model
              4-1 Load the PyFunc model
              4-2 Run predictions and verify results

In [0]:
import mlflow
import mlflow.sklearn
import mlflow.pyfunc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Set basic options for Pandas and Matplotlib

In [0]:
pd.set_option("display.max_columns", None)
%matplotlib inline
plt.style.use('fivethirtyeight')

## Read the feature table from Unity Catalog and prepare test data

We read the same feature table that was used for training, apply the identical
train/test split and scaling, and use the resulting `X_test_scaled` for all
three prediction approaches below. Using the same `random_state=42` guarantees
we are evaluating on exactly the same held-out rows as the model building notebook.

In [0]:
# Unity Catalog coordinates - update these to match your setup
catalog = "book_ai_ml_lakehouse"
schema  = "default"
table   = "hotel_bookings_features"

spark_df   = spark.read.table(f"{catalog}.{schema}.{table}")
bookings_df = spark_df.toPandas()

print(f"Loaded {bookings_df.shape[0]:,} rows and {bookings_df.shape[1]} columns.")
bookings_df.head()

### Define the feature matrix and target variable

In [0]:
X = bookings_df.drop(columns=['is_canceled'])
y = bookings_df['is_canceled']

print(f"Feature matrix: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

### Split into training and test sets

We use the same 80/20 split and random seed as the model building notebook.
This ensures `X_test_scaled` contains exactly the same rows that the model
was evaluated against during training.

In [0]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set:     {X_test.shape}")

### Apply standard scaling

As in the model building notebook, we fit the scaler on training data only
and apply the same transformation to the test data.

In [0]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Scaling complete.")
print(f"X_test_scaled shape: {X_test_scaled.shape}")

## Load the model from the Unity Catalog registry (recommended)

The model building notebook registered the best-performing model (Random Forest)
to Unity Catalog under `{catalog}.{schema}.hotel_cancellation_classifier`.
Loading from the registry is the recommended approach in production because it
decouples the prediction code from any specific run ID. If the model is retrained
and a new version registered, only the version number below needs to change.

The Unity Catalog model URI follows the format:
`models:/<catalog>.<schema>.<model_name>/<version>`

Use `version="latest"` (or omit it for the latest alias) to always load the
most recently registered version.

In [0]:
# Build the Unity Catalog model URI
model_name    = "hotel_cancellation_classifier"
model_uc_path = f"{catalog}.{schema}.{model_name}"

# Load the latest registered version
model_uri = f"models:/{model_uc_path}/latest"
print(f"Model URI: {model_uri}")

### Load the model as a native scikit-learn model

In [0]:
# Load as a native scikit-learn object
# This gives access to all sklearn methods including predict(), predict_proba(),
# and feature_importances_ for tree-based models
model = mlflow.sklearn.load_model(model_uri=model_uri)
print(f"Model type: {type(model).__name__}")

### Run predictions and evaluate results

In [0]:
# Generate predictions on the test set
y_pred = model.predict(X_test_scaled)

# Summarize prediction distribution
num_zeros = (y_pred == 0).sum()
num_ones  = (y_pred == 1).sum()
print(f"Predicted not canceled (0): {num_zeros:,}")
print(f"Predicted canceled     (1): {num_ones:,}")

# Evaluate against ground truth
acc = accuracy_score(y_test, y_pred)
print(f"\nTest accuracy: {acc:.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

### Inspect feature importances

Because the registered model is a Random Forest, we can inspect which features
drive its predictions. This is useful for explaining model behavior to
business stakeholders and for identifying candidates for future feature engineering.

In [0]:
# Extract feature importances from the Random Forest
importances = pd.Series(model.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
top_features.plot(kind='barh', color='steelblue')
plt.gca().invert_yaxis()
plt.title('Top 15 feature importances (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

## Load the model as a Spark UDF

MLflow can wrap any registered model as a Spark User Defined Function (UDF),
allowing it to generate predictions directly on a Spark DataFrame. This approach
is valuable when scoring large datasets at scale in a distributed Spark pipeline,
where converting to pandas first would be impractical due to memory constraints.

### Create the Spark UDF from the model URI

In [0]:
# Create a Spark UDF from the registered model.
# env_manager='virtualenv' tells MLflow to manage Python dependencies using
# a virtual environment rather than conda, which is faster in Databricks.
pyfunc_udf = mlflow.pyfunc.spark_udf(spark, model_uri=model_uri, env_manager='virtualenv')
print("Spark UDF created successfully.")

### Create a Spark DataFrame from the scaled test values

In [0]:
# Convert the scaled numpy array back to a pandas DataFrame, preserving
# the original column names. This is required because spark.createDataFrame()
# on a raw numpy array produces columns named _0, _1, _2... which the model
# does not recognize as its expected feature names.
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_train.columns)

# Convert to a Spark DataFrame
X_test_sp = spark.createDataFrame(X_test_scaled_df)

print(f"Spark DataFrame created with {X_test_sp.count():,} rows and {len(X_test_sp.columns)} columns.")
X_test_sp.printSchema()

### Run predictions with the Spark UDF

In [0]:
from pyspark.sql.functions import struct, col

# Apply the UDF to the Spark DataFrame.
# struct(*X_test_sp.columns) packages all feature columns into a single
# struct so the UDF can process them as a combined model input.
predicted_df = X_test_sp.withColumn('prediction', pyfunc_udf(struct(*X_test_sp.columns)))

display(predicted_df.select('prediction').limit(10))

### Verify the predictions

In [0]:
from pyspark.sql.functions import col

# If the prediction column contains arrays (common with some model flavors),
# extract the first element. If it already contains scalars, this is a no-op.
new_predicted_df = predicted_df.withColumn('prediction', col('prediction')[0])

count_zeros = new_predicted_df.filter(col('prediction') == 0).count()
count_ones  = new_predicted_df.filter(col('prediction') == 1).count()

print(f"Predicted not canceled (0): {count_zeros:,}")
print(f"Predicted canceled     (1): {count_ones:,}")
print(f"Total predictions: {count_zeros + count_ones:,}")

## Load the model as a generic PyFunc model

The PyFunc format is MLflow's framework-agnostic model interface. It provides a
consistent `predict()` API regardless of whether the underlying model is a
scikit-learn estimator, a TensorFlow SavedModel, or any other supported framework.
This makes PyFunc the preferred loading method when building inference pipelines
that need to work across different model types without framework-specific code.

### Load the PyFunc model and run predictions

In [0]:
# Load using the PyFunc interface
pyfunc_model = mlflow.pyfunc.load_model(model_uri=model_uri)
print(f"PyFunc model loaded: {type(pyfunc_model)}")

# PyFunc expects a pandas DataFrame, not a numpy array
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_train.columns)
y_pred_pyfunc = pyfunc_model.predict(X_test_scaled_df)

### Verify the predictions

In [0]:
num_zeros = (y_pred_pyfunc == 0).sum()
num_ones  = (y_pred_pyfunc == 1).sum()

print(f"Predicted not canceled (0): {num_zeros:,}")
print(f"Predicted canceled     (1): {num_ones:,}")

# Verify consistency with the sklearn predictions above
match = (y_pred == y_pred_pyfunc).all()
print(f"\nPyFunc predictions match sklearn predictions: {match}")

## Summary: when to use each loading method

| Method | Best used when |
|---|---|
| `mlflow.sklearn.load_model()` | You need sklearn-specific methods such as `predict_proba()`, `feature_importances_`, or direct access to model attributes |
| `mlflow.pyfunc.spark_udf()` | Scoring large datasets at scale in a Spark pipeline without converting to pandas |
| `mlflow.pyfunc.load_model()` | Building framework-agnostic inference code that should work regardless of the underlying model type |

In all three cases, we load from the Unity Catalog registry URI (`models:/...`) rather
than a hardcoded run ID. This means the prediction code continues to work correctly
when the model is retrained and a new version is registered, without any changes
to this notebook.

## End of notebook